In [ ]:
pip install geopandas rtree libpysal mgwr rioxarray rasterio rasterstats shapely pyproj pandas numpy scikit-learn statsmodels matplotlib contextily


In [ ]:
## Getting all the libraries needed
import warnings
warnings.filterwarnings("ignore")

import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
import rasterio
from rasterstats import zonal_stats
import libpysal
from mgwr.sel_bw import Sel_BW
from mgwr.gwr import GWR
from mgwr.utils import shift_colormap
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import mapclassify
import rioxarray as rxr
import xarray as xr
import contextily as ctx

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Getting Street Tree Census Data from 2015

In [ ]:
trees_df = pd.read_csv("/content/drive/MyDrive/Urban Informatics/Term Project/Data/2015StreetTreeCensus.csv")
# Keeping only needed columns
trees_df = trees_df[['tree_id','latitude','longitude','health','tree_dbh','spc_common','status']]
trees_gdf = gpd.GeoDataFrame(
    trees_df,
    geometry=gpd.points_from_xy(trees_df.longitude, trees_df.latitude),
    crs="EPSG:4326"
).to_crs(epsg=2263)

Census Tract Geometry

In [ ]:
tracts = gpd.read_file("/content/drive/MyDrive/Urban Informatics/Term Project/Data/2020_Census_Tracts_20251203.geojson").to_crs(epsg=2263)
tracts = tracts[['geoid','shape_area','geometry','cdta2020','ct2020','boroct2020','boroname']]
# tracts.head() (check during code writing)

In [ ]:
# Assigning tract numbers to trees and then grouping them by tract
trees_with_tract = gpd.sjoin(
    trees_gdf,
    tracts[["geoid", "geometry"]],
    how="left",
    predicate="within"
)

tree_by_tract = (
    trees_with_tract.groupby("geoid")
    .agg(
        tree_count=("tree_id", "count"),
        avg_dbh=("tree_dbh", "mean"),
        pct_dead=("status", lambda x: (x == "Dead").mean() * 100),
        pct_poor_health=("health", lambda x: (x == "Poor").mean() * 100),
    )
    .reset_index()
)

Getting ACS Variables In

In [ ]:
acs_paths = {
    "poverty": "/content/drive/MyDrive/Urban Informatics/Term Project/Data/2015PercentBelowPoverty.csv",
    "education": "/content/drive/MyDrive/Urban Informatics/Term Project/Data/2015Education.csv",
    "limited_eng": "/content/drive/MyDrive/Urban Informatics/Term Project/Data/2015LimitedEnglish.csv",
    "race": "/content/drive/MyDrive/Urban Informatics/Term Project/Data/2015RaceAgeSex.csv",
}

In [ ]:
# some cleaning
acs_clean = {}

acs_clean["poverty"] = pd.read_csv(acs_paths["poverty"])[[    "GEO_ID",
    "S1702_C02_001E"    # Percent below poverty
]]

acs_clean["education"] = pd.read_csv(acs_paths["education"])[[    "GEO_ID",
    "S1501_C02_015E"    # Percent Bachelor's degree or higher
]]

acs_clean["race"] = pd.read_csv(acs_paths["race"])[[    "GEO_ID",
    "DP05_0032PE",   # White
    "DP05_0033PE",   # Black
    "DP05_0066PE",    # Hispanic/Latino
    "DP05_0028E"     # Total Pop
]]

acs_clean["limited_eng"] = pd.read_csv(acs_paths["limited_eng"])[[    "GEO_ID",
    "S1602_C04_001E"    # percent limited english households
]]

In [ ]:
# renaming stuff so it is more legible
rename_dict = {
    "S1702_C02_001E": "pct_poverty",
    "S1501_C02_015E": "pct_bachelors_up",
    "DP05_0032PE": "pct_white",
    "DP05_0033PE": "pct_black",
    "DP05_0066PE": "pct_hispanic",
    "S1602_C04_001E": "pct_englimit",
    "DP05_0028E": "total_pop",
}

for name, df in acs_clean.items():
    df.rename(columns=rename_dict, inplace=True)

In [ ]:
## need to do the funny thing of separating the last 11 digits of geoid here
def add_geoid(df):
    df["geoid"] = df["GEO_ID"].astype(str).str[-11:]
    df = df.drop(index=[0, 1]).reset_index(drop=True)
    return df

# Apply to all ACS components
for name, df in acs_clean.items():
    acs_clean[name] = add_geoid(df)
for name, df in acs_clean.items():
    acs_clean[name] = add_geoid(df)
    acs_clean[name] = acs_clean[name].drop(columns=["GEO_ID"])

In [ ]:
#merging

acs_merged = (
    acs_clean["poverty"]
    .merge(acs_clean["education"], on="geoid", how="left")
    .merge(acs_clean["race"], on="geoid", how="left")
    .merge(acs_clean["limited_eng"], on="geoid", how="left")
)
# acs_merged.head() (check when I was writing)

Now joining tree data to my census tracts

In [ ]:
trees_gdf["is_poor"] = (trees_gdf["health"] == "Poor").astype(int)
trees_gdf["is_fair"] = (trees_gdf["health"] == "Fair").astype(int)
trees_gdf["is_good"] = (trees_gdf["health"] == "Good").astype(int)
trees_gdf["is_dead"] = (trees_gdf["status"] == "Dead").astype(int)
trees_gdf["is_alive"] = (trees_gdf["status"] == "Alive").astype(int)

In [ ]:
trees_with_tract = gpd.sjoin(
    trees_gdf,
    tracts[["geoid", "geometry"]],
    how="left",
    predicate="within"
)
# trees_with_tract.head() (CHECK)

In [ ]:
# reverting some of these to percentages for normalization
tree_summary = (
    trees_with_tract.groupby("geoid")
    .agg(
        total_trees=("health", "count"),
        poor_health_count=("is_poor", "sum"),
        fair_health_count=("is_fair", "sum"),
        good_health_count=("is_good", "sum"),
        dead_count=("is_dead", "sum")
    )
    .reset_index()
)
tree_summary["pct_poor"] = tree_summary["poor_health_count"] / tree_summary["total_trees"] * 100
tree_summary["pct_fair"] = tree_summary["fair_health_count"] / tree_summary["total_trees"] * 100
tree_summary["pct_good"] = tree_summary["good_health_count"] / tree_summary["total_trees"] * 100
tree_summary["pct_dead"] = tree_summary["dead_count"] / tree_summary["total_trees"] * 100

In [ ]:
# finally merging
tracts["geoid"] = tracts["geoid"].astype(str)
tree_summary["geoid"] = tree_summary["geoid"].astype(str)

tracts = tracts.merge(tree_summary, on="geoid", how="inner")

Merging tracts with ACS now

In [ ]:
tracts = tracts.merge(
    acs_merged,
    left_on="geoid",
    right_on="geoid",
    how="left"
)
# tracts.head() (CHECK --> first row is all NaN values)
# decided to rename and start a new gdf to have the merged tracts file intact to go back to in
gwr_df = tracts.copy() # called gwr dataframe because I hope to use this for further GWR stuff after this project, ignore.

In [ ]:
# realized there is a bit more cleaning for tree stuff
tree_vars = ["pct_poor", "pct_fair", "pct_good", "pct_dead", "total_trees"]
gwr_df[tree_vars] = gwr_df[tree_vars].fillna(0)

Impervious Surfaces Data

In [ ]:
impervious_raster = "/content/drive/MyDrive/Urban Informatics/Term Project/Data/NLCD_mit89sbhqw70gq/Annual_NLCD_FctImp_2015_CU_C1V1_mit89sbhqw70gq.tiff"
raster = rasterio.open(impervious_raster)
raster_crs = raster.crs

gwr_df_raster = gwr_df.to_crs(raster_crs)

# Computing mean impervious land fraction per tract
impervious_stats = zonal_stats(
    vectors=gwr_df_raster["geometry"],
    raster=impervious_raster,
    stats=["mean"],
    nodata=raster.nodata,
    all_touched=True
)

impervious_df = pd.DataFrame(impervious_stats)
impervious_df["geoid"] = gwr_df_raster["geoid"].values
impervious_df.rename(columns={"mean":"impervious_frac"}, inplace=True)

# Merging into main dataframe
if "impervious_frac" in gwr_df.columns:
    gwr_df = gwr_df.drop(columns=["impervious_frac"])
gwr_df = gwr_df.merge(impervious_df[["geoid","impervious_frac"]], on="geoid", how="left")
gwr_df["impervious_frac"] = pd.to_numeric(gwr_df["impervious_frac"], errors="coerce")
gwr_df["impervious_frac"] = gwr_df["impervious_frac"].fillna(gwr_df["impervious_frac"].mean())

# reproject back to NYC State Plane
gwr_df = gwr_df.to_crs(2263)

# quick check (because previous attempts to calculate per census tract raster mean had all NaN values)
print(gwr_df[["geoid","impervious_frac"]].head())

In [ ]:
## Where are the most impervious areas?
top_impervious = (
    gwr_df[["geoid", "impervious_frac"]]
    .sort_values(by="impervious_frac", ascending=False)
    .reset_index(drop=True)
)

print(top_impervious.head(10)) ## Most in Queens, two in Brooklyn, and one in Manhattan

In [ ]:
## First, I want to make sure the imperviousness raster is residualized, as it is highly impacted by histroic demographic patterns.
from sklearn.linear_model import LinearRegression
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.decomposition import PCA

demographics = ["pct_poverty", "pct_bachelors_up", "pct_white", "pct_black", "pct_hispanic", "pct_englimit"]

# Convert demographic columns to numeric, coerce errors to NaN, drop missing stuff
gwr_df_clean = gwr_df.copy()
for col in demographics:
    gwr_df_clean[col] = pd.to_numeric(gwr_df_clean[col], errors='coerce')
gwr_df_clean = gwr_df_clean.dropna(subset=demographics + ["impervious_frac", "pct_poor"])
print(f"Rows after cleaning: {len(gwr_df_clean)}")


Mapping

In [ ]:
boroughs_gdf = gpd.read_file("/content/drive/MyDrive/Urban Informatics/Term Project/Data/Borough_Boundaries_20251205.geojson").to_crs(gwr_df.crs)

In [ ]:
# found a way to make better north arrow

def add_north_arrow(ax, size=500, location=(0.9, 0.1), lw=2):

    import matplotlib.transforms as transforms
    import matplotlib.patches as patches

    trans = transforms.blended_transform_factory(ax.transAxes, ax.transAxes)
    cx, cy = location
    circle = patches.Circle((cx, cy), 0.03, transform=ax.transAxes,
                            fill=False, linewidth=lw, color="black")
    ax.add_patch(circle)
    ax.plot(
        [cx, cx],
        [cy, cy + 0.03],
        transform=ax.transAxes,
        color="black",
        linewidth=lw
    )

In [ ]:
def add_scale_bar(ax, gdf, length_mile=4, location=(0.75, 0.05), linewidth=3):

    if gdf.crs.to_epsg() == 2263:
        units_per_mile = 5280   # feet
        label = f"{length_mile} mile"
    else:
        raise ValueError("Scale bar code is currently only set up for EPSG:2263 (feet).")

    length_units = length_mile * units_per_mile
    xmin, ymin, xmax, ymax = gdf.total_bounds

    x0 = xmin + location[0] * (xmax - xmin)
    x1 = x0 + length_units
    y  = ymin + location[1] * (ymax - ymin)

    ax.plot([x0, x1], [y, y], color="black", linewidth=linewidth)
    ax.text((x0 + x1) / 2, y + 200,f"{length_mile} miles", ha="center",
        va="bottom", fontsize=10)

In [ ]:
def plot_choropleth_categorical(
    df,
    column,
    title=None,
    legend_title="",
    cmap="viridis",
    k=5,
    percent_labels=True,
    export_path=None
):

    df[column] = pd.to_numeric(df[column], errors="coerce")
    classifier = mapclassify.NaturalBreaks(df[column].dropna(), k=k)
    df["_bin"] = classifier.find_bin(df[column])
    df["_bin"] = df["_bin"].astype("category")


    raw_labels = classifier.get_legend_classes(fmt="{:.2f}")
    if percent_labels:
        labels = [lbl + "%" for lbl in raw_labels]
    else:
        labels = raw_labels

    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    df.plot(column="_bin", cmap=cmap, linewidth=0.1, edgecolor="white", ax=ax)
    boroughs_gdf.boundary.plot(ax=ax, color="black", linewidth=1)
    if title is None:
        title = column.replace("_", " ").title()
    ax.set_title(title, fontsize=16)
    add_north_arrow(ax, location=(0.92, 0.12))
    add_scale_bar(ax, df)
    cmap_obj = plt.cm.get_cmap(cmap, len(labels))
    patches = [
        mpatches.Patch(color=cmap_obj(i), label=labels[i])
        for i in range(len(labels))
    ]

    cmap_obj = plt.cm.get_cmap(cmap, len(labels))
    patches = [
        mpatches.FancyBboxPatch(
            (0, 0), 1, 1,
            boxstyle="round,pad=0.2,rounding_size=0.3",  # rounded corners
            facecolor=cmap_obj(i),
            edgecolor="gray"
        )
        for i in range(len(labels))
    ]

    legend = ax.legend(
        patches,
        labels,
        title=legend_title,
        fontsize=11,
        title_fontsize=11,
        frameon=False,
        loc="upper left",
        handlelength=1.8,       # makes color boxes longer
        handleheight=1.8        # taller boxes
    )
    for text in legend.get_texts():
        text.set_text(text.get_text().replace("[", "").replace("]", "").replace("(", ""))

    ax.axis("off")

    plt.show()
    df.drop(columns=["_bin"], inplace=True)

In [ ]:
# double checking before mapping
pct_columns = [col for col in gwr_df_clean.columns if col.startswith("pct_")] + ["impervious_frac"]

for col in pct_columns:
    gwr_df_clean[col] = pd.to_numeric(gwr_df_clean[col], errors="coerce")

In [ ]:
plot_choropleth_categorical(
    gwr_df_clean,
    "impervious_frac",
    title="Impervious Surface Proportions in NYC",
    cmap="Greys"
)

In [ ]:
plot_choropleth_categorical(
    gwr_df_clean,
    "pct_black",
    title="Percent Black Population in NYC",
    cmap="Purples"
)

In [ ]:
plot_choropleth_categorical(
    gwr_df_clean,
    "pct_poverty",
    title="Percent Population in Poverty",
    cmap="Blues"
)

In [ ]:
# I'll end up not using this varible...
plot_choropleth_categorical(
    gwr_df_clean,
    "pct_bachelors_up",
    title="Percent Population with a Bachelor's Degree",
    cmap="BuGn"
)

In [ ]:
plot_choropleth_categorical(
    gwr_df_clean,
    "pct_hispanic",
    title="Percent Hispanic Population in NYC",
    cmap="PuBu"
)

In [ ]:
plot_choropleth_categorical(
    gwr_df_clean,
    "pct_englimit",
    title="Percent Households with Limited English Spoken in NYC",
    cmap="YlGn"
)

In [ ]:
from sklearn.neighbors import KernelDensity

poor_trees = trees_with_tract[trees_with_tract['is_poor'] == 1].copy()
poor_trees.head()

In [ ]:
coords = np.array(list(poor_trees.geometry.apply(lambda p: (p.x, p.y))))

kde = KernelDensity(kernel='gaussian', bandwidth=600)  # bandwidth in CRS units (feet)
kde.fit(coords)

xmin, ymin, xmax, ymax = gwr_df.total_bounds
xx, yy = np.meshgrid(np.linspace(xmin, xmax, 600),
                     np.linspace(ymin, ymax, 600))
grid_coords = np.vstack([xx.ravel(), yy.ravel()]).T
log_dens = kde.score_samples(grid_coords)
density = np.exp(log_dens).reshape(xx.shape)

In [ ]:
kde_xr = xr.DataArray(
    density,
    dims=("y", "x"),
    coords={
        "y": np.linspace(ymin, ymax, density.shape[0]),
        "x": np.linspace(xmin, xmax, density.shape[1])
    },
    name="density"
).rio.write_crs(boroughs_gdf.crs)

boroughs_geom = [feature.__geo_interface__ for feature in boroughs_gdf.geometry]
treehealth_kdeclip = kde_xr.rio.clip(boroughs_geom, boroughs_gdf.crs) ## Clipping the KDE raster to boroughs

fig, ax = plt.subplots(figsize=(12,12))
c = ax.imshow(
    treehealth_kdeclip.values,
    origin="lower",
    cmap="Reds",
    extent=[treehealth_kdeclip.x.min(), treehealth_kdeclip.x.max(), treehealth_kdeclip.y.min(), treehealth_kdeclip.y.max()]
)

gwr_df.boundary.plot(ax=ax, color="black", linewidth=0.2, alpha=0.3)
boroughs_gdf.boundary.plot(ax=ax, color="black", linewidth=1)

fig.colorbar(c, ax=ax, label="Density of Trees in Poor Health")
add_north_arrow(ax)
add_scale_bar(ax, gwr_df_clean)

ax.set_title("Density of Street Trees in Poor Health")
ax.axis("off")
plt.show()

In [ ]:
### Some context: Where are the high poverty areas compared to poor tree health?
threshold = gwr_df_clean["pct_poverty"].quantile(0.90)
high_poverty_layer = gwr_df_clean[gwr_df_clean["pct_poverty"] >= threshold].copy()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))
c = ax.imshow(treehealth_kdeclip, origin='lower', cmap='Reds',
              extent=(xmin, xmax, ymin, ymax))
gwr_df.boundary.plot(ax=ax, color='black', linewidth=0.2, alpha=0.3)

high_poverty_layer.boundary.plot(ax=ax, color="darkblue", linewidth=0.8)
high_poverty_layer.plot(ax=ax, color="blue", alpha=0.10)

fig.colorbar(c, ax=ax, label='Density of Trees in Poor Health')

boroughs_gdf.boundary.plot(ax=ax, color="black", linewidth=1)
add_north_arrow(ax)
add_scale_bar(ax, gwr_df_clean)

ax.set_title("Density of Street Trees in Poor Health 2015")
ax.axis("off")
plt.show()

In [ ]:
### Some context: Where are the highly impervious areas compared to poor tree health?
threshold = gwr_df_clean["impervious_frac"].quantile(0.80)
high_impervious_layer = gwr_df_clean[gwr_df_clean["impervious_frac"] >= threshold].copy()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))
c = ax.imshow(treehealth_kdeclip, origin='lower', cmap='Reds',
              extent=(xmin, xmax, ymin, ymax))
gwr_df.boundary.plot(ax=ax, color='black', linewidth=0.5, alpha=0.3)

high_impervious_layer.boundary.plot(ax=ax, color="darkgreen", linewidth=0.8)
high_impervious_layer.plot(ax=ax, color="green", alpha=0.10)

fig.colorbar(c, ax=ax, label='Density of Trees in Poor Health')

boroughs_gdf.boundary.plot(ax=ax, color="black", linewidth=1)
add_north_arrow(ax)
add_scale_bar(ax, gwr_df_clean)

ax.set_title("Density of Street Trees in Poor Health 2015")
ax.axis("off")
plt.show()

I want to take a different approach now to overlay aggregated social vulnerability related to poor canopy health concentrations.

In [ ]:
# creating vulnerability variables for a quick mcda
vulnerability_vars = [
    "pct_poverty",
    "pct_englimit",
    "pct_black",
    "pct_hispanic"
]

for var in vulnerability_vars:
    min_val = gwr_df_clean[var].min()
    max_val = gwr_df_clean[var].max()
    gwr_df_clean[f"{var}_norm"] = (gwr_df_clean[var] - min_val) / (max_val - min_val)

Now weighing the vulnerability. Poorer, black and hispanic neighborhoods have historically been underserved in terms of plantings, tree canopy cover, and maintenance services. The limited english variable represents the ability with which residents can file complaints/ feel comfortable asking local governments for further maintenance action.

In [ ]:
weights = {
    "pct_poverty_norm": 0.30,
    "pct_englimit_norm": 0.20,
    "pct_black_norm": 0.20,
    "pct_hispanic_norm": 0.20
}

In [ ]:
gwr_df_clean["mcda_score"] = (
    gwr_df_clean["pct_poverty_norm"] * weights["pct_poverty_norm"] +
    gwr_df_clean["pct_englimit_norm"] * weights["pct_englimit_norm"] +
    gwr_df_clean["pct_black_norm"] * weights["pct_black_norm"] +
    gwr_df_clean["pct_hispanic_norm"] * weights["pct_hispanic_norm"]
)

gwr_df_clean["mcda_rank"] = gwr_df_clean["mcda_score"].rank(ascending=False)

print(gwr_df_clean[[ "mcda_score", "mcda_rank"]].head())

Now mapping

In [ ]:
plot_choropleth_categorical(
    gwr_df_clean,
    "mcda_score",
    title="Social Vulnerability Score",
    cmap="Oranges",
)


In [ ]:
### Some context: Where are the highly vulnerable areas compared to poor tree health?
threshold = gwr_df_clean["mcda_score"].quantile(0.90)
highvulnerability_layer = gwr_df_clean[gwr_df_clean["mcda_score"] >= threshold].copy()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))
c = ax.imshow(treehealth_kdeclip, origin='lower', cmap='Reds',
              extent=(xmin, xmax, ymin, ymax))
gwr_df.boundary.plot(ax=ax, color='black', linewidth=0.2, alpha=0.3)

highvulnerability_layer.boundary.plot(ax=ax, color="black", linewidth=0.8)
highvulnerability_layer.plot(ax=ax, color="black", alpha=0.10)

fig.colorbar(c, ax=ax, label='Density of Trees in Poor Health')

boroughs_gdf.boundary.plot(ax=ax, color="black", linewidth=1)
add_north_arrow(ax)
add_scale_bar(ax, gwr_df_clean)
ax.set_title("Density of Street Trees in Poor Health 2015 Versus High Social Vulnerability")
ax.axis("off")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

sc = ax.scatter(
    gwr_df_clean['impervious_frac'],
    gwr_df_clean['poor_health_count'],
    color='darkgreen',
    s=50,
    alpha=0.7,
    edgecolor='none'
)
z = np.polyfit(gwr_df_clean['impervious_frac'], gwr_df_clean['poor_health_count'], 1)
p = np.poly1d(z)
ax.plot(
    gwr_df_clean['impervious_frac'],
    p(gwr_df_clean['impervious_frac']),
    color='black',
    linewidth=2,
    alpha=0.8,
    label=f"y = {z[0]:.2f}x + {z[1]:.1f}"
)

ax.set_xlabel("Impervious Surface Fraction", fontsize=12)
ax.set_ylabel("Trees in Poor Health (Count per Tract)", fontsize=12)
ax.set_title("Impact of Impervious Surface on Tree Canopy Health", fontsize=14)
ax.legend()

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()